In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, LabelEncoder
import re
import joblib 
from tqdm import tqdm

# --- 설정값 ---
MODEL_PATH = "spline_best_lstm_model2.pth"

SCALER_PATH = "spline_scaler2.joblib" 
ENCODER_PATH = "spline_label_encoder2.joblib"

# 테스트 데이터 경로
TEST_BASE_DIR = "./Validation/TS/" 

NUM_FRAMES = 10
NUM_LANDMARKS = 33
NUM_COORDS = 3
FEATURES_PER_FRAME = NUM_LANDMARKS * NUM_COORDS  # 99
TOTAL_FEATURES = NUM_FRAMES * FEATURES_PER_FRAME  # 990
HIDDEN_SIZE = 64  # LSTM 유닛 수 (Keras의 units)
NUM_LAYERS = 2  # LSTM 레이어 수 (단일 레이어 LSTM)
LEARNING_RATE = 0.001
EPOCHS = 50
BATCH_SIZE = 32
DROPOUT_RATE = 0.3
PATIENCE = 10  
# 모델 하이퍼파라미터 (훈련 시와 동일)

# --- 0. 장치 설정 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
try:
    label_encoder = joblib.load(ENCODER_PATH)
    num_classes = len(label_encoder.classes_)
    print(f"저장된 LabelEncoder 로드 완료. 클래스: {label_encoder.classes_}")
except FileNotFoundError:
    print(f"에러: LabelEncoder 파일 '{ENCODER_PATH}'을 찾을 수 없습니다.")
    exit()
except Exception as e:
    print(f"에러: LabelEncoder 로드 중 오류: {e}")
    exit()

try:
    scaler = joblib.load(SCALER_PATH)
    print("저장된 StandardScaler 로드 완료")
except FileNotFoundError:
    print(f"에러: StandardScaler 파일 '{SCALER_PATH}'을 찾을 수 없습니다.")
    exit()
except Exception as e:
    print(f"에러: StandardScaler 로드 중 오류: {e}")
    exit()

저장된 LabelEncoder 로드 완료. 클래스: ['BY' 'FY' 'N' 'SY']
저장된 StandardScaler 로드 완료


In [4]:
class FallDetectionLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout_rate,
        bidirectional=True,
    ):
        super(FallDetectionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=(
                dropout_rate if num_layers > 1 else 0
            ),  
            bidirectional=bidirectional,
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size * self.num_directions, num_classes)

    def forward(self, x):

        lstm_out, _ = self.lstm(x)  # 초기 state 없으면 자동 0으로 처리됨
        last_time_step_out = lstm_out[:, -1, :]

        out = self.dropout(last_time_step_out)
        out = self.fc(out) 

        # Softmax를 통해 각 클래스에 대한 확률로 변환
        out = torch.softmax(out, dim=1)
        return out


def numerical_sort_key(filename_path):
    """Path 객체를 받아 파일 이름 기준 숫자 정렬 키 반환"""
    filename_str = filename_path.name
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split("([0-9]+)", filename_str)
    ]

In [5]:
model = FallDetectionLSTM(
    input_size=FEATURES_PER_FRAME,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=num_classes,
    dropout_rate=DROPOUT_RATE,
    bidirectional=True,  # 양방향 LSTM 사용
).to(device)

try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

    model.eval()

    print(f"모델 가중치 로드 완료: {MODEL_PATH}")

except FileNotFoundError:

    print(f"에러: 모델 파일 '{MODEL_PATH}'을 찾을 수 없습니다.")

    exit()

except Exception as e:

    print(f"에러: 모델 로드 중 오류 발생: {e}")

    exit()

모델 가중치 로드 완료: spline_best_lstm_model2.pth


In [6]:
# --- 4. MediaPipe Pose 초기화 ---
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)


In [8]:
import json
import cv2
import numpy as np
import mediapipe as mp
import os
from pathlib import Path
import re
import pandas as pd
import torch  

# # --- 설정 상수 ---
VALIDATION_TS_DIR = Path("./Validation/TS/")
VALIDATION_LABEL_DIR = Path("./Validation/LABEL/")
CLASS_LABELS = ["BY", "FY", "N", "SY"]  # 학습, 레이블인코더로 함
MIN_DETECTED_FRAMES = 3

# --- PyTorch 모델 관련 설정 ---
INPUT_SIZE = FEATURES_PER_FRAME  # 입력 특징 수
NUM_CLASSES = 4  # 클래스 개수
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용할 디바이스: {device}")

# --- 모델 로딩 ---
try:
    # 1. 모델 구조 인스턴스 생성 
    model = FallDetectionLSTM(
        INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT_RATE, bidirectional=True
    )

    # 2. 저장된 state_dict (가중치) 로드
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

    # 3. 모델을 설정된 디바이스로 이동
    model.to(device)

    # 4. 모델을 평가 모드로 설정 (Dropout, BatchNorm 등 비활성화)
    model.eval()

    print(f"PyTorch 모델 로딩 성공: {MODEL_PATH}")

except FileNotFoundError:
    print(f"모델 파일({MODEL_PATH})을 찾을 수 없습니다. 경로를 확인하세요.")
    exit()
except Exception as e:
    print(f"모델 로딩 중 오류 발생: {e}")
    print(
        "모델 구조 정의(YourLSTMModel)나 파라미터가 실제 모델과 일치하는지 확인하세요."
    )
    exit()

# --- MediaPipe Pose 초기화 ---
mp_pose = mp.solutions.pose
pose_estimator = mp_pose.Pose(
    static_image_mode=True,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)


def extract_pose_sequence(image_paths, pose_estimator):
    FEATURES_PER_FRAME, NUM_FRAMES, VALIDATION_LABEL_DIR, VALIDATION_TS_DIR

    pose_sequence = []
    processed_image_count = 0

    if len(image_paths) != NUM_FRAMES:
        print(
            f"{NUM_FRAMES}개의 이미지가 필요하지만 {len(image_paths)}개가 주어졌습니다."
        )
        return None

    for img_path in image_paths:
        bbox_coords = None
        json_path = None

        try:
            relative_path = img_path.relative_to(VALIDATION_TS_DIR)
            json_filename = img_path.stem + ".json"
            json_path = VALIDATION_LABEL_DIR / relative_path.parent / json_filename
        except Exception:
            json_path = None

        if json_path and json_path.exists():
            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if "bboxdata" in data and "bbox_location" in data["bboxdata"]:
                        bbox_str = data["bboxdata"]["bbox_location"]
                        coords = [float(c.strip()) for c in bbox_str.split(",")]
                        if len(coords) == 4:
                            bbox_coords = coords
            except Exception:
                pass

        try:
            image = cv2.imread(str(img_path))
            if image is None:
                pose_sequence.append([0.0] * FEATURES_PER_FRAME)
                processed_image_count += 1
                continue

            h, w = image.shape[:2]
            image_to_process = image

            if bbox_coords:
                x_min, y_min, x_max, y_max = bbox_coords
                x_min, y_min = max(0, int(x_min)), max(0, int(y_min))
                x_max, y_max = min(w, int(x_max)), min(h, int(y_max))
                if x_min < x_max and y_min < y_max:
                    cropped_image = image[y_min:y_max, x_min:x_max]
                    if cropped_image.size > 0:
                        image_to_process = cropped_image

            image_rgb = cv2.cvtColor(image_to_process, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            result = pose_estimator.process(image_rgb)
            image_rgb.flags.writeable = True

            if result.pose_landmarks:
                landmarks = result.pose_landmarks.landmark
                pose_vec = []
                for lm in landmarks:
                    pose_vec.extend([lm.x, lm.y, lm.z])
                if len(pose_vec) == FEATURES_PER_FRAME:
                    pose_sequence.append(pose_vec)
                else:
                    pose_sequence.append([0.0] * FEATURES_PER_FRAME)
            else:
                pose_sequence.append([0.0] * FEATURES_PER_FRAME)
            processed_image_count += 1

        except Exception as e:
            print(f"이미지 처리/MP 중 예외 ({img_path.name}): {e}. 0 벡터 추가.")
            pose_sequence.append([0.0] * FEATURES_PER_FRAME)
            processed_image_count += 1
            continue

    if len(pose_sequence) == NUM_FRAMES:
        return np.array(pose_sequence)
    else:
        return None  # 사실상 위쪽 len 체크에서 걸러짐


# --- 메인 처리 루프 ---
if __name__ == "__main__":
    print("--- 포즈 추출, 필터링, 보간 및 PyTorch 예측 시작 ---")
    print("-" * 30)

    total_sequences = 0
    processed_sequences = 0
    filtered_out_sequences = 0
    interpolated_sequences = 0
    predicted_sequences = 0
    prediction_results = {}

    for class_label in CLASS_LABELS:
        class_dir = VALIDATION_TS_DIR / class_label
        if not class_dir.is_dir():
            continue

        print(f"\n클래스 처리 중: {class_label} ({class_dir})")
        sequence_folders = sorted([d for d in class_dir.iterdir() if d.is_dir()])

        for seq_folder_path in sequence_folders:
            total_sequences += 1
            print(f" -> 시퀀스 처리 중: {seq_folder_path.name}")

            image_files = list(seq_folder_path.glob("*.jpg"))
            if not image_files:
                image_files = list(seq_folder_path.glob("*.png"))

            try:
                image_files.sort(key=numerical_sort_key)
            except Exception as e:
                print(
                    f"{seq_folder_path.name} 이미지 정렬 오류: {e}. 건너<0xEB><0x9B><0x84>니다."
                )
                prediction_results[str(seq_folder_path)] = (
                    class_label,
                    "FileSortError",
                )
                continue

            raw_pose_data = extract_pose_sequence(image_files, pose_estimator)

            if raw_pose_data is None:
                print(
                    f"시퀀스 처리 불가 (이미지 수 오류 등): {seq_folder_path.name}"
                )
                prediction_results[str(seq_folder_path)] = (
                    class_label,
                    "SequenceError",
                )
                continue

            processed_sequences += 1

            detected_mask = np.any(raw_pose_data != 0.0, axis=1)
            num_detected_frames = np.sum(detected_mask)
            print(f"감지된 프레임 수: {num_detected_frames} / {NUM_FRAMES}")

            if num_detected_frames < MIN_DETECTED_FRAMES:
                print(
                    f"필터링됨: 감지 프레임 수가 기준({MIN_DETECTED_FRAMES}) 미만."
                )
                filtered_out_sequences += 1
                prediction_results[str(seq_folder_path)] = (class_label, "FilteredOut")
                continue

            # --- 보간 ---
            pose_data_for_prediction = raw_pose_data.copy()
            needs_interpolation = np.any(raw_pose_data == 0.0)

            if needs_interpolation:
                print(f"보간 필요: 0 벡터(결측 프레임) 포함.")
                interpolated_sequences += 1
                sequence_with_nan = raw_pose_data.copy()
                sequence_with_nan[sequence_with_nan == 0.0] = np.nan
                df_seq = pd.DataFrame(sequence_with_nan)
                df_interpolated = df_seq.interpolate(
                    method="linear", axis=0, limit_direction="both"
                )
                df_interpolated = df_interpolated.fillna(0.0)
                pose_data_for_prediction = df_interpolated.values
                print(f"보간 완료.")

            # --- PyTorch 모델 예측 ---
            print(f"필터링/보간 완료. PyTorch 예측 준비.")
            predicted_label = "PredictionError"  # 기본값

            # 예측 시에는 그래디언트 계산 불필요
            with torch.no_grad():
                try:
                    input_tensor = (
                        torch.from_numpy(pose_data_for_prediction)
                        .unsqueeze(0)
                        .float()
                        .to(device)
                    )

                    # 2. 모델 예측 수행
                    outputs = model(input_tensor)  # 모델의 forward 메소드 호출
                    predicted_index = torch.argmax(outputs, dim=1).item()

                    # 4. 인덱스를 클래스 레이블로 변환
                    predicted_label = CLASS_LABELS[predicted_index]

                except Exception as e:
                    print(f"PyTorch 예측 중 오류 발생: {e}")

            # 결과 저장
            true_label = class_label
            prediction_results[str(seq_folder_path)] = (true_label, predicted_label)
            if predicted_label != "PredictionError":
                predicted_sequences += 1  # 성공적으로 예측된 경우만 카운트
            print(
                f"  🏁 시퀀스: {seq_folder_path.name} | 실제: {true_label} | 예측: {predicted_label}"
            )

    # --- 리소스 정리 ---
    print("\n--- 리소스 해제 중 ---")
    pose_estimator.close()
    print("MediaPipe Pose Estimator가 닫혔습니다.")

    # --- 결과 요약  ---
    print("\n--- 처리 요약 ---")
    print(f"총 발견된 시퀀스 수: {total_sequences}")
    print(f"포즈 추출 시도된 시퀀스 수: {processed_sequences}")
    print(f"오류로 처리 불가 시퀀스 수: {total_sequences - processed_sequences}")
    print(
        f"필터링되어 제외된 시퀀스 수 (<{MIN_DETECTED_FRAMES} 프레임 감지): {filtered_out_sequences}"
    )
    print(f"보간이 수행된 시퀀스 수: {interpolated_sequences}")
    print(f"최종 예측이 수행된 시퀀스 수: {predicted_sequences}")  # 예측 성공 기준

    # 정확도 계산 (예측 수행되고 오류가 없었던 시퀀스 기준)
    correct_predictions = 0
    valid_prediction_attempts = 0
    real_chk = 0

    for seq_path, (true, pred) in prediction_results.items():
        if pred not in [
            "SequenceError",
            "FileSortError",
            "FilteredOut",
            "PredictionError",
        ]:
            valid_prediction_attempts += 1
            if true == pred:
                correct_predictions += 1
            if len(true) == len(pred):
                real_chk += 1

    if valid_prediction_attempts > 0:
        accuracy = (correct_predictions / valid_prediction_attempts) * 100
        real_accuracy = (real_chk / valid_prediction_attempts) * 100
        print(f"\n--- 예측 정확도 (최종 예측된 시퀀스 기준) ---")
        print(f"올바른 예측 수: {correct_predictions}")
        print(f"총 유효 예측 수: {valid_prediction_attempts}")
        print(f"정확도: {accuracy:.2f}%")
        print(f"정확도: {real_accuracy:.2f}%")
    else:
        print("\n정확도를 계산할 유효한 예측이 없습니다.")

    print("\n--- 스크립트 종료 ---")

사용할 디바이스: cuda
PyTorch 모델 로딩 성공: spline_best_lstm_model2.pth
--- 포즈 추출, 필터링, 보간 및 PyTorch 예측 시작 ---
------------------------------

클래스 처리 중: BY (Validation\TS\BY)
 -> 시퀀스 처리 중: 00074_H_A_BY_C1
감지된 프레임 수: 9 / 10
보간 필요: 0 벡터(결측 프레임) 포함.
보간 완료.
필터링/보간 완료. PyTorch 예측 준비.
  🏁 시퀀스: 00074_H_A_BY_C1 | 실제: BY | 예측: BY
 -> 시퀀스 처리 중: 00074_H_A_BY_C2
감지된 프레임 수: 8 / 10
보간 필요: 0 벡터(결측 프레임) 포함.
보간 완료.
필터링/보간 완료. PyTorch 예측 준비.
  🏁 시퀀스: 00074_H_A_BY_C2 | 실제: BY | 예측: BY
 -> 시퀀스 처리 중: 00074_H_A_BY_C3
감지된 프레임 수: 9 / 10
보간 필요: 0 벡터(결측 프레임) 포함.
보간 완료.
필터링/보간 완료. PyTorch 예측 준비.
  🏁 시퀀스: 00074_H_A_BY_C3 | 실제: BY | 예측: BY
 -> 시퀀스 처리 중: 00074_H_A_BY_C4
감지된 프레임 수: 6 / 10
보간 필요: 0 벡터(결측 프레임) 포함.
보간 완료.
필터링/보간 완료. PyTorch 예측 준비.
  🏁 시퀀스: 00074_H_A_BY_C4 | 실제: BY | 예측: BY
 -> 시퀀스 처리 중: 00074_H_A_BY_C5
감지된 프레임 수: 5 / 10
보간 필요: 0 벡터(결측 프레임) 포함.
보간 완료.
필터링/보간 완료. PyTorch 예측 준비.
  🏁 시퀀스: 00074_H_A_BY_C5 | 실제: BY | 예측: BY
 -> 시퀀스 처리 중: 00074_H_A_BY_C6
감지된 프레임 수: 7 / 10
보간 필요: 0 벡터(결측 프레임) 포함.
보간 완료.
필터링/보간 완료. PyTorc